In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F

# Initialize local SparkSession
spark = (
    SparkSession.builder
    .appName("02_Silver_Transformation")
    .getOrCreate()
)

BRONZE_PATH = "/home/jovyan/data_lake/bronze"
SILVER_PATH = "/home/jovyan/data_lake/silver"
Path(SILVER_PATH).mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).isoformat()
dq_summary = []

def apply_dq_and_quarantine(
    df: DataFrame,
    dataset_name: str,
    valid_condition,
    dedup_cols: list,
    target_name: str,
) -> None:
    """Validate rows, isolate invalid records to quarantine, and write clean Parquet."""
    total_rows = df.count()

    valid_df = df.filter(valid_condition).dropDuplicates(dedup_cols)
    quarantine_df = (
        df.filter(~valid_condition)
          .withColumn("_dataset", F.lit(dataset_name))
          .withColumn("_quarantined_at", F.lit(RUN_TS))
          .withColumn("_reason", F.lit("Failed Silver DQ validation rule"))
    )

    valid_count = valid_df.count()
    quarantine_count = quarantine_df.count()

    valid_df.write.mode("overwrite").parquet(f"{SILVER_PATH}/{target_name}")

    if quarantine_count > 0:
        quarantine_df.select("_dataset", "_quarantined_at", "_reason").write.mode(
            "append"
        ).parquet(f"{SILVER_PATH}/quarantine_rows")

    dq_summary.append({
        "dataset": dataset_name,
        "total_rows": total_rows,
        "valid_rows": valid_count,
        "quarantined_rows": quarantine_count,
        "pass_rate_pct": round((valid_count / total_rows) * 100, 2) if total_rows else 0.0,
        "checked_at": RUN_TS,
    })
    print(f"[DQ] {dataset_name}: {valid_count}/{total_rows} valid ({quarantine_count} quarantined)")

# Transform World Bank Indicators
wb_raw = spark.read.parquet(f"{BRONZE_PATH}/bronze_wb_indicators")

wb_std = wb_raw.select(
    F.upper(F.trim(F.col("country_iso3"))).alias("country_code"),
    F.trim(F.col("country_name")).alias("country_name"),
    F.trim(F.col("indicator_id")).alias("indicator_id"),
    F.trim(F.col("indicator_name")).alias("indicator_name"),
    F.col("year").cast("int").alias("year"),
    F.col("value").cast("double").alias("indicator_value"),
    F.col("_source_system").alias("source_system"),
)

apply_dq_and_quarantine(
    df=wb_std,
    dataset_name="wb_indicators",
    valid_condition=(
        (F.col("country_code") == "EGY")
        & F.col("year").isNotNull()
        & (F.col("year") >= 1980)
        & F.col("indicator_value").isNotNull()
    ),
    dedup_cols=["country_code", "indicator_id", "year"],
    target_name="clean_wb_indicators",
)

# Save Data Quality Report Summary
dq_df = spark.createDataFrame(pd.DataFrame(dq_summary))
dq_df.write.mode("append").parquet(f"{SILVER_PATH}/dq_report_summary")

print("=== Silver Data Quality Summary ===")
spark.read.parquet(f"{SILVER_PATH}/dq_report_summary").show(truncate=False)

spark.stop()